# Galerie explicabilité — 02 · Expliquer TON modèle 🟠

> Étagère asynchrone **optionnelle**. Pas de livrable, pas de note.
>
> ⏱️ ~1 h 30 · 🟠 **peu guidé** : les cellules sont des `# TODO`. Aucun corrigé
> n'est donné — mais chaque étape a un encadré **🧭 repères** qui te dit à quoi
> ressemble un bon résultat.

## Ce qu'on te demande

Refaire, seul, sur un modèle **réel** le geste du notebook 01 : produire une
explication locale et une explication globale, puis en tirer **trois phrases
utilisables dans une note client**.

Le notebook 01 t'a montré le geste sur un jeu fabriqué exprès. Ici il n'y a plus
de règle génératrice connue, plus de dataset conçu pour bien se comporter : c'est
la situation de ton cas d'usage certif.

**Prérequis** : avoir fait `01_explicabilite_pas_a_pas.ipynb`.

## Choisis ton support

| Option | Support | Pour qui |
|---|---|---|
| **A** (recommandée) | `models/pyrenex_risk_v2.joblib` + `data/reference_set.csv` de ton repo **M6-B2** (les mêmes fichiers existent dans M5-B1) | tout le monde : c'est un modèle que tu as déjà exploité en production |
| **B** | le modèle de **ton cas d'usage certif** | si ton pipeline est déjà entraîné et sauvegardé |
| **C** (repli) | le jeu `donnees_aubrac.py` du notebook 01, régénéré avec `random_state=7` et `n=6000` | si tu n'as plus les repos M5/M6 sous la main |

> ⚠️ **Environnement — lis ceci avant de perdre 20 minutes.**
> `pyrenex_risk_v2.joblib` a été sérialisé avec **scikit-learn 1.5.1**. Chargé
> depuis une version plus récente (1.8 par exemple), il émet d'abord un
> `InconsistentVersionWarning`, puis échoue franchement :
>
> ```
> AttributeError: Can't get attribute '_RemainderColsList' on
> <module 'sklearn.compose._column_transformer'>
> ```
>
> **La bonne pratique, et la solution ici** : travailler dans le venv du repo
> M6-B2 (où scikit-learn est figé), et y ajouter `pip install shap`. Un modèle
> sérialisé n'est pas portable entre versions de bibliothèque — c'est aussi
> pour ça qu'on versionne l'environnement à côté du modèle (cf. M5).

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap
import sklearn

# Option A : adapte le chemin vers TON clone du repo M6-B2
RACINE = Path("../../ia-dev-id-parcours-m6-b2")
CHEMIN_MODELE = RACINE / "models" / "pyrenex_risk_v2.joblib"
CHEMIN_DONNEES = RACINE / "data" / "reference_set.csv"
COLONNE_CIBLE = "loan_status"

print("scikit-learn", sklearn.__version__, "| shap", shap.__version__)
print("modèle présent :", CHEMIN_MODELE.exists(), "| données présentes :", CHEMIN_DONNEES.exists())

## Étape 1 — Ouvrir la boîte

Charge le modèle et les données. Avant toute chose : **de quoi est fait ce
`.joblib` ?** Un `Pipeline` n'est pas un modèle, c'est un enchaînement. Tu dois
savoir nommer chacune de ses étapes et retrouver l'estimateur final.

In [ ]:
# TODO
# 1. charger le modèle et les données
# 2. afficher le type de l'objet chargé et le nom de ses étapes
# 3. isoler le préprocesseur d'un côté, l'estimateur final de l'autre
# 4. séparer X (features) et y (cible), puis calculer les probabilités prédites

modele = ...
donnees = ...

> 🧭 **Repères (étape 1)**
> - Un `Pipeline` de 2 étapes, dont l'estimateur final est un
>   `RandomForestClassifier` (200 arbres, `max_depth=10`).
> - 1 500 lignes, 15 colonnes dont la cible.
> - La cible est **textuelle** (`Fully Paid` / `Charged Off`) : à toi de décider
>   comment tu la traites, et le fichier `pyrenex_risk_v2.json` te dit dans quel
>   sens (`target_mapping`). Ne devine pas ce genre de chose.
> - Si `predict_proba` te renvoie une erreur de colonnes, c'est que tu as laissé
>   la cible dans `X`.

## Étape 2 — Retrouver des noms lisibles

SHAP va travailler sur les colonnes **après** transformation : one-hot compris.
Il te faut donc la liste des noms de sortie du préprocesseur, et un jeu de
données transformé.

In [ ]:
# TODO
# 1. récupérer les noms des colonnes en sortie du préprocesseur
# 2. produire la matrice transformée qui sera donnée à SHAP
# 3. vérifier que le nombre de noms == le nombre de colonnes de la matrice

> 🧭 **Repères (étape 2)** — tu dois obtenir **43 colonnes** en sortie pour
> 14 colonnes d'entrée. Si tu n'as que 14 noms, tu as pris les colonnes d'entrée
> et pas celles de sortie ; l'explication sera décalée d'une variable à l'autre
> et tu ne t'en apercevras pas.

## Étape 3 — Les valeurs SHAP, et la preuve qu'elles sont justes

Calcule les contributions avec l'explainer adapté à un modèle à base d'arbres,
puis **vérifie l'additivité** sur un dossier : `valeur de base + somme des
contributions` doit redonner exactement la probabilité prédite. Tant que cette
égalité ne tombe pas juste, ne va pas plus loin : ton explication porte sur autre
chose que ton modèle.

In [ ]:
# TODO

> 🧭 **Repères (étape 3)**
> - Le calcul prend quelques secondes sur 1 500 dossiers. Si tu attends des
>   minutes, tu n'as pas pris `TreeExplainer`.
> - Classifieur binaire → tu récupères un tableau à 3 dimensions ; la dernière
>   est la classe. Tu veux la classe **1** (`Charged Off`, le défaut).
> - L'écart entre `base + somme` et la probabilité prédite doit être de l'ordre
>   de 1e-9, pas de 0.05.

## Étape 4 — Deux dossiers, deux explications locales

Choisis **le dossier le plus risqué** et **un dossier clairement sain**, et
produis leurs waterfalls. Pense à afficher des valeurs métier lisibles plutôt
que les valeurs standardisées (piège n°2 du notebook 01).

In [ ]:
# TODO

## Étape 5 — La vue globale

Produis le classement global (contribution absolue moyenne) et le beeswarm.

In [ ]:
# TODO

> 🧭 **Repères (étape 5)** — sur `reference_set.csv`, les quatre variables de
> tête sont `dti`, `int_rate`, `fico_range_low` et `revol_util`, suivies des
> modalités de `grade`. Si ton classement est dominé par `purpose_*` ou
> `home_ownership_*`, reprends l'étape 2 : tes noms sont probablement mal
> alignés sur tes colonnes.
>
> 🎯 **La question qui compte** : `int_rate` (le taux accordé) est un des
> premiers facteurs de risque selon le modèle. Or ce taux a été **fixé par la
> banque en fonction du risque estimé à l'octroi**. Qu'est-ce que ça t'apprend
> sur ce que le modèle a vraiment appris ? Écris ta réponse en trois lignes —
> c'est typiquement le genre de remarque qui fait la différence en soutenance.

## Étape 6 — Le livrable

Rédige, en markdown dans la cellule suivante, **trois phrases** directement
collables dans une note au client :

1. une phrase **globale** : sur quoi ce modèle s'appuie, et ce qu'on en pense ;
2. une phrase **locale** : pourquoi le dossier n°X reçoit un avis défavorable,
   en français, sans jargon ni nom de colonne ;
3. une phrase de **limite** : ce que cette explication ne dit pas.

Contrainte : aucune de ces phrases ne contient le mot « SHAP ».

*(à toi d'écrire — remplace ce texte)*

1. …
2. …
3. …

## ⭐ Étape 7 (optionnelle) — L'audit de proxy

Le notebook 01 a montré qu'une variable anodine peut reconstruire un attribut
sensible. `reference_set.csv` ne contient pas d'attribut sensible directement
exploitable — mais ton **cas d'usage** en contient peut-être un, et ton client
en aura sûrement un.

Reprends la démarche du § 5 du notebook 01 sur un jeu de ton choix :
croise les contributions SHAP d'une variable suspecte avec un attribut sensible
que tu n'as **pas** donné au modèle, puis mesure l'écart de taux de décision
entre groupes et compare-le à la règle des 4/5 (M2-B2).

In [ ]:
# TODO (optionnel)

## ✅ Checklist de sortie

- [ ] J'ai su nommer les étapes d'un pipeline sérialisé et isoler l'estimateur final.
- [ ] J'ai vérifié l'additivité avant de commenter quoi que ce soit.
- [ ] Mes waterfalls affichent des valeurs métier, pas des valeurs standardisées.
- [ ] Mon top 5 global correspond aux repères — sinon j'ai trouvé pourquoi.
- [ ] J'ai écrit mes trois phrases sans le mot « SHAP ».
- [ ] Je sais dire ce que cette explication **ne** prouve **pas**.
- [ ] J'ai reporté ma conclusion dans la ligne « explicabilité » de ma grille C4.